# Podcast-to-Production Agent — ADK Testing Notebook

Walk the full pipeline step by step: parse a script, run the Director, Researcher and Audio Producer agents, then inspect the production report.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
from dotenv import load_dotenv
load_dotenv('../.env')
print('project:', os.getenv('GOOGLE_CLOUD_PROJECT'))

## 1. Parse the demo script

In [ ]:
from src.phase2_document_processing.pdf_parser import PDFScriptParser
from src.phase2_document_processing.speaker_identifier import SpeakerIdentifier
from src.phase2_document_processing.script_analyzer import ScriptAnalyzer

parser = PDFScriptParser()
script_data = parser.parse('../static/demo_script.pdf')
script_data['genre'] = 'technology'
print(script_data['speakers'], script_data['mood'], script_data['estimated_duration'])
script_data['dialogue_segments'][:3]

In [ ]:
profiles = SpeakerIdentifier().identify(script_data['dialogue_segments'])
analysis = ScriptAnalyzer().analyze(script_data)
profiles, analysis['music_direction'], analysis['production_notes']

## 2. Director agent

In [ ]:
from src.phase4_adk_agents.director_agent import DirectorAgent
director = DirectorAgent()
director_notes = director.run(script_data)
director_notes

## 3. Researcher agent (Parallel Search)

In [ ]:
from src.phase4_adk_agents.researcher_agent import ResearcherAgent
research = ResearcherAgent().run(script_data, director_notes)
research

## 4. Audio producer (Gemini TTS + Lyria 3)

In [ ]:
from src.phase4_adk_agents.audio_producer_agent import AudioProducerAgent
audio = AudioProducerAgent().run(script_data, director_notes)
audio['music_path'], audio['total_segments']

In [ ]:
from src.tools.audio_utils import concatenate, mix_with_music, duration_seconds
paths = [a['audio_path'] for a in audio['audio_files'] if a['audio_path']]
dialogue = concatenate(paths, '../outputs/dialogue.wav')
episode = mix_with_music(dialogue, audio['music_path'], '../outputs/episode.wav')
episode, duration_seconds(episode) if episode else None

## 5. Full orchestration

In [ ]:
from src.phase4_adk_agents.orchestrator import PodcastOrchestrator
report = PodcastOrchestrator().process_script('../static/demo_script.pdf', genre='technology')
report['recommendations']